# 驗證 Ollama /api/chat 是否可用且回傳 message.content

本 Notebook 用來在本機直接測試 Ollama 的 /api/chat 端點：
- 檢查服務可連線（/api/tags）
- 發送最小可行的 chat 請求
- 確認 HTTP 200，並解析出 response.message.content
- 印出耗時與關鍵回應欄位，便於除錯

可調參數：BASE_URL、MODEL、KEEP_ALIVE、TIMEOUT。

In [5]:
import os
import sys
import json
import time
import platform
from urllib.parse import urljoin

import urllib.request
import urllib.error

# 可調參數
BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434/")
MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b-instruct-q4_0")
KEEP_ALIVE = os.environ.get("OLLAMA_KEEP_ALIVE", "10m")
TIMEOUT = float(os.environ.get("OLLAMA_TIMEOUT", 30.0))

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Config:", {
    "BASE_URL": BASE_URL,
    "MODEL": MODEL,
    "KEEP_ALIVE": KEEP_ALIVE,
    "TIMEOUT": TIMEOUT,
})

def log(msg, **kwargs):
    ts = time.strftime('%H:%M:%S')
    extra = (" " + json.dumps(kwargs, ensure_ascii=False)) if kwargs else ""
    print(f"[{ts}] {msg}{extra}")

Python: 3.11.6 (v3.11.6:8b6ee5ba3b, Oct  2 2023, 11:18:21) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-16.0-arm64-arm-64bit
Config: {'BASE_URL': 'http://127.0.0.1:11434/', 'MODEL': 'qwen2.5:7b-instruct-q4_0', 'KEEP_ALIVE': '10m', 'TIMEOUT': 30.0}


In [6]:
def http_get(url: str, timeout: float = TIMEOUT):
    req = urllib.request.Request(url, method="GET")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = resp.read()
            return resp.status, data
    except urllib.error.HTTPError as e:
        return e.code, e.read()
    except urllib.error.URLError as e:
        return None, str(e).encode()


def http_post(url: str, payload: dict, timeout: float = TIMEOUT):
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Content-Type": "application/json"
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = resp.read()
            return resp.status, data
    except urllib.error.HTTPError as e:
        return e.code, e.read()
    except urllib.error.URLError as e:
        return None, str(e).encode()

In [7]:
# 健康檢查：/api/tags
start = time.time()
status, data = http_get(urljoin(BASE_URL, "/api/tags"))
elapsed = (time.time() - start) * 1000
log("GET /api/tags", status=status, elapsed_ms=round(elapsed, 1))

if status != 200:
    print("Raw:", data[:2000])
    raise RuntimeError("Ollama /api/tags not healthy or unreachable")
else:
    try:
        js = json.loads(data)
        print("tags count:", len(js.get("models", [])))
    except Exception:
        print("Non-JSON or unexpected body, first 2KB:\n", data[:2000])

[09:17:52] GET /api/tags {"status": 200, "elapsed_ms": 4.4}
tags count: 2


In [8]:
# 最小可行 chat 測試
payload = {
    "model": MODEL,
    "keep_alive": KEEP_ALIVE,
    "stream": False,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant that translates between Chinese and English."},
        {"role": "user", "content": "Translate to English: 你好，世界。"}
    ],
    # 與 App 對齊的推論參數
    "options": {
        "temperature": 0.2,
        "num_predict": 128
    }
}

start = time.time()
status, data = http_post(urljoin(BASE_URL, "/api/chat"), payload)
elapsed = (time.time() - start) * 1000
log("POST /api/chat", status=status, elapsed_ms=round(elapsed, 1))

if status != 200:
    print("Raw:", data[:2000])
    raise RuntimeError("/api/chat HTTP status is not 200")

js = json.loads(data)
# 解析非串流回應格式
message = js.get("message") or {}
content = message.get("content")

if not content:
    # 有些實作可能將 text 放在不同欄位，這裡保守輸出原文以便除錯
    print("Body:", json.dumps(js, ensure_ascii=False)[:2000])
    raise RuntimeError("response.message.content is empty")

print("content:", content[:300])

[09:17:55] POST /api/chat {"status": 200, "elapsed_ms": 671.7}
content: Hello, world.


In [ ]:
def run_test(user_text: str, direction: str = "zh2en"):
    if direction == "zh2en":
        prompt = f"Translate to English: {user_text}"
    else:
        prompt = f"翻譯成繁體中文（zh-TW）：{user_text}"

    payload = {
        "model": MODEL,
        "keep_alive": KEEP_ALIVE,
        "stream": False,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant that translates between Chinese and English. When the target language is Chinese, always respond in Traditional Chinese (zh-TW) only."},
            {"role": "user", "content": prompt}
        ],
        "options": {
            "temperature": 0.2,
            "num_predict": 128
        }
    }

    start = time.time()
    status, data = http_post(urljoin(BASE_URL, "/api/chat"), payload)
    elapsed = (time.time() - start) * 1000
    log("POST /api/chat", status=status, elapsed_ms=round(elapsed, 1), direction=direction)

    if status != 200:
        print("Raw:", data[:2000])
        raise RuntimeError("/api/chat HTTP status is not 200")

    js = json.loads(data)
    message = js.get("message") or {}
    content = message.get("content")
    if not content:
        print("Body:", json.dumps(js, ensure_ascii=False)[:2000])
        raise RuntimeError("response.message.content is empty")

    print("OK:", content[:300])

# 範例測試
run_test("今天天氣很好，適合散步。", "zh2en")
run_test("It is a good day for a walk.", "en2zh")

[09:18:11] POST /api/chat {"status": 200, "elapsed_ms": 1428.2, "direction": "zh2en"}
OK: The weather is very nice today, suitable for taking a walk.
[09:18:12] POST /api/chat {"status": 200, "elapsed_ms": 708.3, "direction": "en2zh"}
OK: 这是一个散步的好日子。
[09:18:12] POST /api/chat {"status": 200, "elapsed_ms": 708.3, "direction": "en2zh"}
OK: 这是一个散步的好日子。
